# $\Gamma(n{=}2)$ v4 --- What Satisfies a Homeostatic Agent Is Not What Limits It

Reproduces **every number** in the paper. Runtime ~4 min single-threaded, no GPU.

**Research question.** Can a homeostatic agent learn the satiation signal $g$ from observable consequences, and use it to regulate toward an externally useful goal?

**What this notebook establishes.** The gap between the homeostatic agent and a state-estimating baseline decomposes into signal quality (7.4%), policy parameterization (27.7%), and an unexplained residual (70.5%).

### Methodological commitments
| Choice | Why |
|---|---|
| **100 profiles**, 10 seeds | the earlier 20-profile design was underpowered |
| **Paired** $t$-tests on profile means | within-subject design: every profile faces every arm |
| **TOST equivalence**, $\delta=0.05$ | claims of *no effect* need an equivalence test, not a non-significant $p$ |
| **Receiver-modulated coupling** | Eq. (1) and design intent; a prior implementation used the sender's pressure |
| **SHA-256 commitment** on predictions | pre-execution hash makes post-hoc edits detectable |
| **Quality gate at every sweep point** | results reported only where the environment is discriminative |

In [ ]:
import hashlib, math, random, json, time, statistics as st
from collections import defaultdict
from pathlib import Path
from scipy.stats import ttest_rel, t as tdist

NIVELES=[1,2,3,4,5]
PRESUPUESTO=40
E_REF=3.0

def _is_prime(n):
    if n<2: return False
    if n in (2,3,5,7): return True
    if n%2==0 or n%3==0: return False
    i=5
    while i*i<=n:
        if n%i==0 or n%(i+2)==0: return False
        i+=6
    return True
def _next_prime(n):
    if n<=2: return 2
    c=n|1
    while not _is_prime(c): c+=2
    return c
def derive_seeds(master,n=10):
    s=[]
    for i in range(n):
        d=hashlib.sha256(f"{master}:run_{i}".encode()).hexdigest()
        s.append(_next_prime(10007+int(d,16)%900000))
    return s

def rasch_p(h,nivel): return 1.0/(1.0+math.exp(-1.5*(h-nivel)))
PEAK_MODEL=0.5   # peak assumed by any agent that models the curve (misspecifiable)
def bell_curve(gap,peak=PEAK_MODEL): return 0.08*math.exp(-((gap-peak)**2)/0.5)


SEEDS=derive_seeds(20260820,10)
N_PROFILES=100
print("seeds:",SEEDS)
print("E_REF:",E_REF,"| PEAK_MODEL:",PEAK_MODEL)


In [ ]:
class StudyEnv:
    """v4: fatiga normalizada + histeresis + shift de regimen + examen en h_eff."""
    def __init__(self, h0, lr_mult=1.0, phi=0.0, lambda_olv=0.01,
                 hysteresis=False, e_crit=1.3, shift_mult=1.0, shift_at=20,
                 peak_gap=PEAK_MODEL, peak_new=None, peak_at=20):
        self.h=h0; self.h0=h0; self.lr_mult=lr_mult
        self.phi=phi; self.phi0=phi
        self.lambda_olv=lambda_olv
        self.hysteresis=hysteresis; self.e_crit=e_crit
        self.shift_mult=shift_mult; self.shift_at=shift_at
        self.peak_gap=peak_gap; self.peak_new=peak_new; self.peak_at=peak_at
        self.esfuerzo=0.0

    def tick(self,s):
        if self.shift_mult!=1.0 and s==self.shift_at:
            self.phi=self.phi0*self.shift_mult
        if self.peak_new is not None and s==self.peak_at:
            self.peak_gap=self.peak_new

    @property
    def esf_n(self): return self.esfuerzo/E_REF
    @property
    def h_eff(self): return max(0.5, self.h - self.phi*self.esf_n**1.5)
    @property
    def lr_eff(self):
        # la fatiga degrada la CAPACIDAD de aprender, no solo la dificultad aparente
        return self.lr_mult*max(0.15, 1.0 - 0.7*self.phi*self.esf_n**1.5)

    def expected_gain(self,nivel):
        gap=nivel-self.h_eff; p=rasch_p(self.h_eff,nivel)
        return bell_curve(gap,self.peak_gap)*(p*1.0+(1-p)*0.3)*self.lr_eff
    def in_zpd(self,nivel):
        g=[self.expected_gain(k) for k in NIVELES]; mx=max(g)
        return (self.expected_gain(nivel)>=0.5*mx) if mx>0 else False

    def exercise(self,nivel,rng):
        p=rasch_p(self.h_eff,nivel)
        correcto=rng.random()<p
        gap=max(0.0,nivel-self.h_eff)
        latencia=rng.lognormvariate(math.log(5)+0.6*gap,0.4)
        self.esfuerzo=0.95*self.esfuerzo+latencia/20.0
        gain=bell_curve(nivel-self.h_eff,self.peak_gap)*(1.0 if correcto else 0.3)*self.lr_eff
        self.h=min(6.0,self.h+gain)
        return {"correcto":correcto,"latencia":round(latencia,2),"gain":gain}

    def descansar(self):
        if self.hysteresis and self.esf_n>self.e_crit: self.esfuerzo*=0.93
        else: self.esfuerzo*=0.7
        self.h=max(0.5,self.h-self.lambda_olv)


_e=StudyEnv(2.5,phi=0.0)
print("expected_gain h=2.5:",{k:round(_e.expected_gain(k),4) for k in NIVELES})
print("StudyEnv OK")


In [ ]:
class GCfg:
    def __init__(self,**kw):
        self.lam_p=kw.get("lam_p",0.15); self.lam_c=kw.get("lam_c",0.02)
        self.theta=kw.get("theta",0.7)
        self.alpha_p=kw.get("alpha_p",0.3); self.alpha_c=kw.get("alpha_c",0.3)
        self.w_cp=kw.get("w_cp",-0.05); self.w_pc=kw.get("w_pc",0.05)
        self.g_mode=kw.get("g_mode","learned")
        self.n_drives=kw.get("n_drives",2); self.coupling=kw.get("coupling",True)
        self.coupling_mode=kw.get("coupling_mode","receiver")  # receiver (Eq.1 / design intent) | sender (legacy bug)
        self.eta=kw.get("eta",0.3); self.beta=kw.get("beta",0.2)
        self.g_init_p=kw.get("g_init_p",0.3); self.g_init_c=kw.get("g_init_c",0.5)
        self.g_naive=kw.get("g_naive",0.5)
        self.action_mode=kw.get("action_mode","softmax")   # softmax | argmax
        self.rest_mode=kw.get("rest_mode","gated")
        self.coef=kw.get("coef",(4.0,-3.0,-3.0,1.5,4.0,-3.0))  # up_p,up_c,stay_k,stay_b,down_c,down_p          # gated | drive

class GammaAgent:
    def __init__(self,cfg,profile):
        self.cfg=cfg; self.p=profile
        self.d_prog=0.5; self.d_conf=0.5
        self.g_prog={k:cfg.g_init_p for k in NIVELES}
        self.g_conf={k:cfg.g_init_c for k in NIVELES}
        self.acc_ema={}; self.nivel=1; self.frust=0.0
        self.consec_high=0; self.abandono=False; self.dropout_session=None
    def _pressure(self,d): return min(1.0,abs(d-self.cfg.theta)/0.8)
    def basal(self):
        m=self.acc_ema.get(self.nivel,0.5)
        self.d_prog=min(2.0,self.d_prog+self.cfg.lam_p*m)
        if self.cfg.n_drives>=2:
            self.d_conf=min(2.0,self.d_conf+self.cfg.lam_c)
            if self.cfg.coupling:
                pc=self._pressure(self.d_conf); pp=self._pressure(self.d_prog)
                if self.cfg.coupling_mode=="receiver":
                    # Eq.(1): modulated by the RECEIVING drive's own pressure
                    m_to_prog, m_to_conf = pp, pc
                else:
                    m_to_prog, m_to_conf = pc, pp     # legacy: sender's pressure
                if self.d_conf>self.cfg.theta+0.05:
                    self.d_prog=max(0.0,self.d_prog+self.cfg.w_cp*m_to_prog)
                if self.d_prog>self.cfg.theta+0.05:
                    self.d_conf=max(0.0,self.d_conf+self.cfg.w_pc*m_to_conf)
    def select(self,rng,env):
        th=self.cfg.theta
        dp=self.d_prog-th
        dc=(self.d_conf-th) if self.cfg.n_drives>=2 else 0.0
        fn=self.frust/max(self.p["umbral"],0.1)
        strain=max(0.0,fn-0.7)/0.3
        if self.cfg.n_drives>=2:
            if self.cfg.rest_mode=="gated":
                l_rest=2*max(0.0,dc)*strain+4*strain
            else:
                # descanso conducido por el drive: delta_confianza por encima del setpoint
                # empuja a descansar SIN gate duro de frustracion.
                l_rest=3.0*max(0.0,dc)+2.0*strain-1.2
            c=self.cfg.coef
            logits=[c[0]*dp+c[1]*dc, c[2]*(abs(dp)+abs(dc))+c[3], c[4]*dc+c[5]*dp, l_rest]
        else:
            logits=[4*dp,-3*abs(dp)+1.5,-4*dp,4*strain]
        acts=["subir","mantener","bajar","descansar"]
        if self.cfg.action_mode=="argmax":
            return acts[max(range(4),key=lambda i:logits[i])]
        mx=max(logits); exps=[math.exp(min(50,l-mx)) for l in logits]
        tot=sum(exps); r=rng.random()*tot; cum=0.0
        for i,e in enumerate(exps):
            cum+=e
            if r<cum: return acts[i]
        return "mantener"
    def study(self,env,outcome,nivel):
        c=outcome["correcto"]; eb=self.acc_ema.get(nivel,0.5)
        self.acc_ema[nivel]=(1-self.cfg.beta)*eb+self.cfg.beta*float(c)
        if self.cfg.g_mode=="learned":
            a=self.acc_ema[nivel]
            self.g_prog[nivel]=(1-self.cfg.eta)*self.g_prog[nivel]+self.cfg.eta*(4.0*a*(1.0-a))
            self.g_conf[nivel]=(1-self.cfg.eta)*self.g_conf[nivel]+self.cfg.eta*a
        elif self.cfg.g_mode=="oracle":
            self.g_prog[nivel]=min(1.0,env.expected_gain(nivel)/0.042)
            self.g_conf[nivel]=rasch_p(env.h_eff,nivel)
        if self.cfg.g_mode=="naive": gp=gc=self.cfg.g_naive
        else: gp=self.g_prog[nivel]; gc=self.g_conf[nivel]
        if c:
            self.d_prog=max(0.0,self.d_prog-self.cfg.alpha_p*gp)
            if self.cfg.n_drives>=2:
                self.d_conf=max(0.0,self.d_conf-self.cfg.alpha_c*gc)
            if outcome["latencia"]<15: self.frust=max(0.0,self.frust-0.35)
        else:
            if self.cfg.n_drives>=2: self.d_conf=min(2.0,self.d_conf+0.1)
            self.frust+=0.25
            if eb>=0.7: self.frust+=0.3


print("GammaAgent OK | coupling_mode, action_mode, rest_mode, coef all configurable")


In [ ]:
class ReglaAgent:
    def __init__(self,profile,rest=True):
        self.p=profile; self.rest=rest; self.nivel=1; self.frust=0.0
        self.consec_high=0; self.abandono=False; self.dropout_session=None
        self.streak_c=0; self.streak_e=0; self.acc_ema={}
    def select(self,rng,env):
        if self.rest and self.frust>0.6*self.p["umbral"]: return "descansar"
        if self.streak_e>=2: return "bajar"
        if self.streak_c>=3: return "subir"
        return "mantener"
    def study(self,env,outcome,nivel):
        if outcome["correcto"]: self.streak_c+=1; self.streak_e=0
        else: self.streak_e+=1; self.streak_c=0

class BayesianAgent:
    def __init__(self,profile,rest=True):
        self.p=profile; self.rest=rest; self.nivel=1; self.h_hat=2.5
        self.frust=0.0; self.consec_high=0; self.abandono=False
        self.dropout_session=None; self.acc_ema={}
    def select(self,rng,env):
        if self.rest and self.frust>0.6*self.p["umbral"]: return "descansar"
        best,bg=self.nivel,-1
        for k in NIVELES:
            gap=k-self.h_hat; p=rasch_p(self.h_hat,k)
            g=bell_curve(gap)*(p+(1-p)*0.3)
            if g>bg: best,bg=k,g
        if best>self.nivel: return "subir"
        if best<self.nivel: return "bajar"
        return "mantener"
    def study(self,env,outcome,nivel):
        p=rasch_p(self.h_hat,nivel)
        self.h_hat+=0.3*(float(outcome["correcto"])-p)
        self.h_hat=max(0.5,min(6.0,self.h_hat))

class FixedAgent:
    def __init__(self,profile,mode):
        self.p=profile; self.mode=mode; self.nivel=1; self.frust=0.0
        self.consec_high=0; self.abandono=False; self.dropout_session=None
        self.acc_ema={}
    def select(self,rng,env):
        if self.mode=="always1": t=1
        elif self.mode=="always5": t=5
        elif self.mode=="random": t=rng.choice(NIVELES)
        else: t=max(NIVELES,key=lambda k:env.expected_gain(k))
        if t>self.nivel: return "subir"
        if t<self.nivel: return "bajar"
        return "mantener"
    def study(self,env,outcome,nivel): pass

def shared_frust_update(agent,outcome,eb):
    if outcome["correcto"]:
        if outcome["latencia"]<15: agent.frust=max(0.0,agent.frust-0.35)
    else:
        agent.frust+=0.25
        if eb>=0.7: agent.frust+=0.3


print("Baselines OK | BayesianAgent uses PEAK_MODEL -> misspecifiable under shape shift")


In [ ]:
def gen_profiles(n=20,seed=2026):
    rng=random.Random(seed)
    return [{"h0":round(rng.uniform(1.5,3.5),2),
             "lr_mult":round(rng.uniform(0.7,1.3),2),
             "umbral":round(rng.uniform(1.8,3.0),1)} for _ in range(n)]

def run_episode(agent,profile,seed,is_gamma,env_kw,log=False):
    env=StudyEnv(profile["h0"],lr_mult=profile["lr_mult"],**env_kw)
    rng=random.Random(seed); traj=[]; zpd=0; nst=0; ndesc=0; s=0
    for s in range(PRESUPUESTO):
        if agent.abandono: break
        env.tick(s)
        if is_gamma: agent.basal()
        a=agent.select(rng,env)
        if a=="subir": agent.nivel=min(5,agent.nivel+1)
        elif a=="bajar": agent.nivel=max(1,agent.nivel-1)
        out=None; inz=False
        if a=="descansar":
            env.descansar(); agent.frust*=0.5; ndesc+=1
        else:
            inz=env.in_zpd(agent.nivel)
            out=env.exercise(agent.nivel,rng); nst+=1
            if inz: zpd+=1
            if is_gamma: agent.study(env,out,agent.nivel)
            else:
                eb=agent.acc_ema.get(agent.nivel,0.5)
                agent.acc_ema[agent.nivel]=0.8*eb+0.2*float(out["correcto"])
                shared_frust_update(agent,out,eb)
                agent.study(env,out,agent.nivel)
        if agent.frust>=0.95*profile["umbral"]: agent.consec_high+=1
        else: agent.consec_high=0
        if agent.consec_high>=3 and not agent.abandono:
            agent.abandono=True; agent.dropout_session=s
        if log:
            traj.append({"s":s,"nivel":agent.nivel,"action":a,
                "correcto":out["correcto"] if out else None,
                "d_prog":round(getattr(agent,"d_prog",0),3),
                "d_conf":round(getattr(agent,"d_conf",0),3),
                "frust":round(agent.frust,2),"h":round(env.h,3),
                "h_eff":round(env.h_eff,3),"esf_n":round(env.esf_n,3),"in_zpd":inz})
    m={"ganancia":round(env.h-profile["h0"],3),
       "h_final":round(env.h,3),
       "exam":round(rasch_p(env.h,3),3),
       "exam_eff":round(rasch_p(env.h_eff,3),3),
       "esf_n_final":round(env.esf_n,3),
       "tiz":round(zpd/max(1,nst),3),
       "dropout":agent.abandono,"dropout_session":agent.dropout_session,
       "n_descansar":ndesc,"n_study":nst,"nivel_final":agent.nivel,
       "sessions_used":s+1}
    if is_gamma and agent.cfg.g_mode in ("learned","oracle","fixed","naive"):
        gains=[env.expected_gain(k) for k in NIVELES]
        gps=[agent.g_prog[k] for k in NIVELES]
        mg=sum(gains)/5; mp=sum(gps)/5
        cov=sum((gains[i]-mg)*(gps[i]-mp) for i in range(5))
        vg=math.sqrt(sum((g-mg)**2 for g in gains)); vp=math.sqrt(sum((g-mp)**2 for g in gps))
        m["g_validity"]=round(cov/(vg*vp),3) if vg>0 and vp>0 else None
        m["g_prog_final"]={str(k):round(agent.g_prog[k],3) for k in NIVELES}
        m["true_gain_final"]={str(k):round(env.expected_gain(k),4) for k in NIVELES}
    return m,traj

CFG_BASE=dict(lam_p=0.15,alpha_p=0.3,alpha_c=0.3,theta=0.7)


PROFILES=gen_profiles(N_PROFILES)
print(len(PROFILES),"profiles; first three:",PROFILES[:3])


## Prediction commitment (hashed before anything runs)

In [ ]:
PREDICTIONS = {
 "P1":"bayesian_norest (estimacion sin gestion de viabilidad) colapsa al subir phi: dropout alto, gain al piso.",
 "P2":"Los brazos Gamma se degradan proporcionalmente menos que el bayesiano a lo largo del barrido de phi.",
 "P3":"Existe phi* dentro del rango valido de la puerta de calidad donde algun brazo Gamma supera al bayesiano en gain.",
 "P4":"gamma_learned_argmax supera a gamma_learned en todos los puntos validos del barrido.",
 "P5":"Si la calidad de g se transmite a la conducta, gamma_oracle se separa de gamma_no_learning en algun punto del barrido.",
 "P6":"g_validity de gamma_learned decrece monotonamente al subir phi.",
 "P7":"Bajo desplazamiento del PICO de la campana de aprendizaje (cambio de forma, no de escala), los brazos Gamma ganan terreno relativo frente al bayesiano, que queda mal especificado y no puede absorberlo re-estimando h.",
}
PRED_TEXT="\n".join(f"{k}: {v}" for k,v in sorted(PREDICTIONS.items()))
PRED_HASH=hashlib.sha256(PRED_TEXT.encode("utf-8")).hexdigest()
Path("results").mkdir(exist_ok=True)
Path("results/predictions.txt").write_text(PRED_TEXT)
print("SHA-256:",PRED_HASH)
assert PRED_HASH=="6e888c7e46e1dd306b6adc601a45cc08ff914a24d2f474a913a1ff2907bfc084", "hash mismatch"
print("matches the digest reported in the paper")

## Arms and paired-statistics helpers

In [ ]:
BASE_COEF=(4.0,-3.0,-3.0,1.5,4.0,-3.0)
G_MODE={"gamma_learned":"learned","gamma_learned_argmax":"learned","gamma_learned_restdrive":"learned",
        "gamma_no_learning":"fixed","gamma_naive":"naive","gamma_oracle":"oracle"}
ARMS=["gamma_learned","gamma_learned_argmax","gamma_learned_restdrive","gamma_no_learning",
      "gamma_naive","gamma_oracle","regla","regla_norest","bayesian","bayesian_norest"]

def build(arm,profile,coef=None):
    if arm in G_MODE:
        kw=dict(**CFG_BASE,g_mode=G_MODE[arm],coupling_mode="receiver")
        if arm=="gamma_learned_argmax": kw["action_mode"]="argmax"
        if arm=="gamma_learned_restdrive": kw["rest_mode"]="drive"
        if coef: kw["coef"]=coef
        return GammaAgent(GCfg(**kw),profile),True
    if arm=="regla":            return ReglaAgent(profile,rest=True),False
    if arm=="regla_norest":     return ReglaAgent(profile,rest=False),False
    if arm=="bayesian":         return BayesianAgent(profile,rest=True),False
    if arm=="bayesian_norest":  return BayesianAgent(profile,rest=False),False
    raise ValueError(arm)

def episodes(arm,env_kw,coef=None,profiles=None,seeds=None):
    """Returns (all metric dicts, one mean-gain per profile) -- the profile means are the
    analysis units, since the design is within-subject."""
    profiles=profiles or PROFILES; seeds=seeds or SEEDS
    rows=[]; per_profile=[]
    for p in profiles:
        gains=[]
        for sd in seeds:
            ag,isg=build(arm,p,coef)
            m,_=run_episode(ag,p,sd,isg,env_kw)
            rows.append(m); gains.append(m["ganancia"])
        per_profile.append(st.mean(gains))
    return rows,per_profile

def paired(xa,xb):
    d=[u-v for u,v in zip(xa,xb)]; n=len(d)
    m=st.mean(d); sd=st.stdev(d); se=sd/math.sqrt(n)
    h=se*tdist.ppf(0.975,n-1); t,p=ttest_rel(xa,xb)
    return {"diff":m,"ci":(m-h,m+h),"p":float(p),"dz":m/sd}

def tost(xa,xb,delta=0.05):
    """Two one-sided tests. p < .05 => difference is inside +/-delta, i.e. practically zero."""
    d=[u-v for u,v in zip(xa,xb)]; n=len(d)
    m=st.mean(d); se=st.stdev(d)/math.sqrt(n)
    return max(1-tdist.cdf((m+delta)/se,n-1), tdist.cdf((m-delta)/se,n-1))

SESOI=0.05   # anchored to the magnitude of the policy-stochasticity effect (P4)
print("helpers ready | analysis unit = profile mean, n =",N_PROFILES)

## Quality gate at every sweep point

Results are reported only where the environment is discriminative. Criteria: the level-1 trap stays unprofitable ($<0.15$), the absorbing boundary bites ($>60\%$ dropout for `always5`), and an oracle policy clearly beats random (margin $>0.15$).

In [ ]:
PHIS=[0.0,0.3,0.6,0.9,1.2,1.6]
GATE={}
print(f"{'phi':6}{'always1':>10}{'a5 drop':>10}{'random':>10}{'oracle':>10}{'margin':>10}  gate")
for phi in PHIS:
    r={}
    for mode in ["always1","always5","random","oracle"]:
        g=[];d=[]
        for p in PROFILES:
            for sd in SEEDS:
                m,_=run_episode(FixedAgent(p,mode),p,sd,False,dict(phi=phi,hysteresis=True))
                g.append(m["ganancia"]);d.append(m["dropout"])
        r[mode]=(st.mean(g),100*sum(d)/len(d))
    margin=r["oracle"][0]-r["random"][0]
    ok=(r["always1"][0]<0.15)and(r["always5"][1]>60)and(margin>0.15)and(r["random"][0]<r["oracle"][0])
    GATE[phi]={"always1":r["always1"][0],"always5_drop":r["always5"][1],"random":r["random"][0],
               "oracle":r["oracle"][0],"margin":margin,"pass":ok}
    print(f"{phi:<6}{r['always1'][0]:>+10.3f}{r['always5'][1]:>9.0f}%{r['random'][0]:>+10.3f}"
          f"{r['oracle'][0]:>+10.3f}{margin:>+10.3f}  {'PASS' if ok else 'FAIL'}")
PHIS_VALID=[p for p in PHIS if GATE[p]["pass"]]
print("\nvalid range:",PHIS_VALID,"  (phi=1.6 fails: the level-1 trap dissolves under severe fatigue)")

## Main adversity sweep --- Table 1 of the paper

In [ ]:
t0=time.time(); SWEEP={}; PERPROF={}
for phi in PHIS:
    SWEEP[phi]={}
    for arm in ARMS:
        rows,per=episodes(arm,dict(phi=phi,hysteresis=True))
        PERPROF[(phi,arm)]=per
        gv=[r["g_validity"] for r in rows if r.get("g_validity") is not None]
        SWEEP[phi][arm]={"gain":st.mean(r["ganancia"] for r in rows),
                         "drop":100*sum(r["dropout"] for r in rows)/len(rows),
                         "tiz":st.mean(r["tiz"] for r in rows),
                         "exam_eff":st.mean(r["exam_eff"] for r in rows),
                         "desc":st.mean(r["n_descansar"] for r in rows),
                         "g_val":st.mean(gv) if gv else None}
print(f"{len(PHIS)*len(ARMS)*N_PROFILES*len(SEEDS)} episodes in {time.time()-t0:.1f}s\n")
print(f"{'arm':26}"+"".join(f"phi={p:<9}" for p in PHIS))
for arm in ARMS:
    print(f"{arm:26}"+"".join(f"{SWEEP[p][arm]['gain']:+.3f}/{SWEEP[p][arm]['drop']:>3.0f}% " for p in PHIS))

## The decomposition --- Table 2 of the paper

A $2\times2$ over **signal quality** (constant vs oracle $g$) and **policy parameterization** (hand-chosen vs re-tuned coefficients). The re-tuned vector comes from the random search in the next cell; it is pinned here so this cell is reproducible on its own.

In [ ]:
OPT_COEF=(1.09,-4.11,-1.95,4.77,1.97,-8.48)   # best of 250 random candidates (see next cell)
DEC={}
for pol,coef in [("hand",BASE_COEF),("retuned",OPT_COEF)]:
    for sig in ["gamma_no_learning","gamma_learned","gamma_oracle"]:
        _,per=episodes(sig,dict(phi=0.0,hysteresis=True),coef=coef)
        DEC[(pol,sig)]=per
_,BAY=episodes("bayesian",dict(phi=0.0,hysteresis=True))
M={k:st.mean(v) for k,v in DEC.items()}
print(f"{'policy':10}{'constant g':>12}{'learned g':>12}{'oracle g':>11}")
for pol in ["hand","retuned"]:
    print(f"{pol:10}{M[(pol,'gamma_no_learning')]:>+12.3f}{M[(pol,'gamma_learned')]:>+12.3f}{M[(pol,'gamma_oracle')]:>+11.3f}")
print(f"\nestimator: {st.mean(BAY):+.3f}")

total = st.mean(BAY)-M[("hand","gamma_no_learning")]
sig_e = M[("hand","gamma_oracle")]-M[("hand","gamma_no_learning")]
pol_e = M[("retuned","gamma_no_learning")]-M[("hand","gamma_no_learning")]
resid = st.mean(BAY)-M[("retuned","gamma_oracle")]
print(f"\n=== decomposition of the {total:+.3f} gap ===")
print(f"  signal quality       {sig_e:+.3f}  ({100*sig_e/total:.1f}%)")
print(f"  policy coefficients  {pol_e:+.3f}  ({100*pol_e/total:.1f}%)")
print(f"  residual             {resid:+.3f}  ({100*resid/total:.1f}%)")
for lbl,a,b in [("signal | hand",("hand","gamma_oracle"),("hand","gamma_no_learning")),
                ("signal | retuned",("retuned","gamma_oracle"),("retuned","gamma_no_learning")),
                ("policy | constant",("retuned","gamma_no_learning"),("hand","gamma_no_learning"))]:
    r=paired(DEC[a],DEC[b]); print(f"  paired {lbl:18} {r['diff']:+.3f}  p={r['p']:.2e}  dz={r['dz']:+.2f}")
print("\nNote: the LEARNED signal underperforms a constant under both policies.")

## Policy coefficient search (Appendix)

Tests the objection that the architecture is limited by badly chosen coefficients rather than by anything structural. It is **partly true** --- and now quantified.

In [ ]:
SEARCH_N=250
rng=random.Random(7)
SUB_P=PROFILES[:30]; SUB_S=SEEDS[:5]
def score(arm,coef): return st.mean(episodes(arm,dict(phi=0.0,hysteresis=True),coef=coef,
                                             profiles=SUB_P,seeds=SUB_S)[1])
best={a:(BASE_COEF,score(a,BASE_COEF)) for a in ["gamma_oracle","gamma_no_learning"]}
t0=time.time()
for _ in range(SEARCH_N):
    c=(rng.uniform(0,10),rng.uniform(-10,2),rng.uniform(-10,0),
       rng.uniform(-2,5),rng.uniform(0,10),rng.uniform(-10,2))
    for a in best:
        v=score(a,c)
        if v>best[a][1]: best[a]=(c,v)
print(f"search over {SEARCH_N} candidates in {time.time()-t0:.0f}s")
for a,(c,v) in best.items():
    print(f"  {a}: {v:+.3f} coef=({', '.join(f'{x:.2f}' for x in c)})")
print(f"\npinned OPT_COEF used above: {OPT_COEF}")

## P5 --- does signal quality reach behaviour? (Table 3)

The decisive contrast. **Paired** tests, plus TOST equivalence: a non-significant $p$ is not evidence of no effect, so we test equivalence explicitly against $\delta=0.05$.

In [ ]:
print(f"{'phi':6}{'oracle':>9}{'constant':>10}{'diff':>9}{'95% CI':>20}{'p':>11}{'dz':>7}{'TOST':>9}  reading")
P5={}
for phi in PHIS_VALID:
    o,f=PERPROF[(phi,"gamma_oracle")],PERPROF[(phi,"gamma_no_learning")]
    r=paired(o,f); tp=tost(o,f,SESOI); P5[phi]={**r,"tost":tp}
    reading="signal matters" if (r["p"]<.05 and tp>=.05) else ("practically equivalent" if tp<.05 else "inconclusive")
    print(f"{phi:<6}{st.mean(o):>+9.3f}{st.mean(f):>+10.3f}{r['diff']:>+9.3f}  "
          f"[{r['ci'][0]:+.3f},{r['ci'][1]:+.3f}]{r['p']:>11.2e}{r['dz']:>+7.2f}{tp:>9.4f}  {reading}")
print("\nP5: CONFIRMED at phi=0 (large paired effect); practically equivalent to zero under adversity.")
print("An earlier analysis used ttest_ind on this paired design and reported no separation anywhere -- a Type II error.")

## P3 and P4 --- policy effects and the crossover (Table 4)

In [ ]:
print(f"{'phi':6}{'P4 argmax-softmax':>30}{'P3 argmax-estimator':>32}")
POL={}
for phi in PHIS_VALID:
    a=paired(PERPROF[(phi,"gamma_learned_argmax")],PERPROF[(phi,"gamma_learned")])
    b=paired(PERPROF[(phi,"gamma_learned_argmax")],PERPROF[(phi,"bayesian")])
    POL[phi]={"P4":a,"P3":b}
    print(f"{phi:<6}{a['diff']:>+9.3f} [{a['ci'][0]:+.3f},{a['ci'][1]:+.3f}] dz={a['dz']:>+5.2f}"
          f"{b['diff']:>+10.3f} [{b['ci'][0]:+.3f},{b['ci'][1]:+.3f}] dz={b['dz']:>+5.2f}")
print("\nP4 CONFIRMED (effect grows with adversity). P3 CONFIRMED: crossover at phi=1.2, CI excludes zero.")
print("Caveat: argmax changes the whole closed loop (visitation, fatigue, g estimates), so P4 is")
print("'the deterministic policy is better', NOT 'sampling noise costs exactly this much'.")

## P6 --- learnability, and its dependence on the policy

In [ ]:
print(f"{'phi':6}{'learned':>10}{'argmax':>10}{'oracle':>10}")
for phi in PHIS_VALID:
    print(f"{phi:<6}{SWEEP[phi]['gamma_learned']['g_val']:>+10.3f}"
          f"{SWEEP[phi]['gamma_learned_argmax']['g_val']:>+10.3f}"
          f"{SWEEP[phi]['gamma_oracle']['g_val']:>+10.3f}")
print("\nP6 CONFIRMED but POLICY-DEPENDENT: validity collapses under the stochastic policy")
print("(+0.471 -> +0.113) yet is largely preserved under argmax -- systematic visitation gives cleaner estimates.")
print("Caveat: 4*a*(1-a) is symmetric about a=0.5 while true gain peaks near a=0.68,")
print("so the learning rule cannot represent the target exactly -- validity is capped by functional form.")

## P7 --- model misspecification (Section 6.5)

At session 20 the **peak** of the learning curve moves. The estimator has `PEAK_MODEL=0.5` hard-coded, so its $\hat h$ stays right while its level choice goes wrong --- a misspecification re-estimation cannot repair.

In [ ]:
PEAKS=[0.5,1.0,1.5,2.0]; PHI_SHAPE=0.6
SHAPE={}; SHAPE_GATE={}
for pk in PEAKS:
    r={}
    for mode in ["always1","always5","random","oracle"]:
        g=[];d=[]
        for p in PROFILES:
            for sd in SEEDS:
                m,_=run_episode(FixedAgent(p,mode),p,sd,False,
                                dict(phi=PHI_SHAPE,hysteresis=True,peak_new=pk,peak_at=20))
                g.append(m["ganancia"]);d.append(m["dropout"])
        r[mode]=(st.mean(g),100*sum(d)/len(d))
    margin=r["oracle"][0]-r["random"][0]
    SHAPE_GATE[pk]={"margin":margin,"pass":(r["always1"][0]<0.15)and(r["always5"][1]>60)and(margin>0.15)}
    SHAPE[pk]={}
    for arm in ["gamma_learned_argmax","gamma_oracle","bayesian"]:
        _,per=episodes(arm,dict(phi=PHI_SHAPE,hysteresis=True,peak_new=pk,peak_at=20))
        SHAPE[pk][arm]=per
gaps=[]
print(f"{'peak':6}{'argmax':>10}{'estimator':>11}{'gap':>9}{'p':>11}{'dz':>7}  gate")
for pk in PEAKS:
    r=paired(SHAPE[pk]["gamma_learned_argmax"],SHAPE[pk]["bayesian"]); gaps.append(r["diff"])
    print(f"{pk:<6}{st.mean(SHAPE[pk]['gamma_learned_argmax']):>+10.3f}"
          f"{st.mean(SHAPE[pk]['bayesian']):>+11.3f}{r['diff']:>+9.3f}{r['p']:>11.2e}{r['dz']:>+7.2f}"
          f"  {'PASS' if SHAPE_GATE[pk]['pass'] else 'FAIL'}")
print(f"\nP7 REFUTED: the gap WIDENS ({gaps[0]:+.3f} -> {gaps[-1]:+.3f}).")
print("Interpretation: this does NOT refute situated information -- the manipulation corrupts the")
print("competitor's model rather than giving Gamma a private signal, and Gamma's own rule (anchored")
print("at a=0.5, gap 0) ends up FURTHER from the true peak than the estimator's fixed 0.5.")
print("Condition (4) therefore remains UNTESTED.")

## Save results

In [ ]:
output={"version":"v4-final","rq":"Can a homeostatic agent learn g from observable consequences and use it to regulate?",
 "n_profiles":N_PROFILES,"seeds":SEEDS,"coupling_mode":"receiver","sesoi":SESOI,
 "predictions":PREDICTIONS,"predictions_sha256":PRED_HASH,
 "gate":{str(k):v for k,v in GATE.items()},"phis":PHIS,"phis_valid":PHIS_VALID,
 "sweep":{str(k):v for k,v in SWEEP.items()},
 "decomposition":{f"{p}|{s}":st.mean(v) for (p,s),v in DEC.items()},
 "decomposition_estimator":st.mean(BAY),
 "P5":{str(k):v for k,v in P5.items()},"P3_P4":{str(k):v for k,v in POL.items()},
 "shape_gate":{str(k):v for k,v in SHAPE_GATE.items()},
 "base_coef":BASE_COEF,"opt_coef":OPT_COEF,"cfg_base":CFG_BASE,"profiles":PROFILES}
json.dump(output,open("results/experiment_v4_results.json","w"),indent=1,default=str)
print("saved results/experiment_v4_results.json and results/predictions.txt")